<a href="https://colab.research.google.com/github/Danial-071/langchain-project/blob/main/LangChain_Tools_and_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install langchain-openrouter
# !pip install langchain-classic
# !pip install langchain-community
# !pip install langchain-core
!pip install wikipedia

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11757 sha256=58f4f85c56264a4d108f653b5e6304d3903e7e33f2b7cf59644a63ce231fd92e
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


In [ ]:
from langchain_openrouter import ChatOpenRouter
from langchain_classic.agents import Tool
from langchain.agents import create_agent
from langchain_core.tools import tool
from google.colab import userdata
import os
os.environ['OPENROUTER_API_KEY'] = userdata.get("API")

model = ChatOpenRouter( model = userdata.get('model'))

In [ ]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str):
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]


    result = sum(numbers)
    return {"result": result}

In [ ]:
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object",add_tool)

add_numbers("1 2")

tool object name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x7c664ca52020>


{'result': 3}

In [ ]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


Tool Name:
AddTool
Tool Description:
Adds a list of numbers and returns the result.
Tool Function:
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x7c664ca52020>)>


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 a b"
print(add_tool.invoke(test_input))  # Example

Calling Tool Function:
{'result': 60}


In [ ]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs: str) -> dict:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return {"result": sum(numbers)}

In [ ]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description)
print("Args: \n", add_numbers.args)


Name: 
 add_numbers
Description: 
 Adds all numeric values found in the input string.
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [ ]:
test_input = "what is the sum between 10, 20 and 30 "
print(add_numbers.invoke(test_input))  # Example

{'result': 60.0}


In [ ]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Tool Constructor Approach:
Has Schema: True


@tool Decorator Approach:
Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [ ]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float).
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

In [ ]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return sum(numbers)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[add_tool]
)

In [ ]:
# Use the agent
response = agent.invoke({
    "messages": [("human", "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")]
})
print(response["messages"][-1].content)

The combined GDP of the United States, Canada, and Mexico in 2023 is:

\[
27.72\ \text{trillion} + 2.14\ \text{trillion} + 1.79\ \text{trillion} = \mathbf{31.65\ \text{trillion\ USD}}.
\]


In [ ]:
agent_2 = create_agent(
    model=,
    tools=[sum_numbers_from_text]
)
response = agent_2.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response["messages"][-1].content)

The absolute values of -10, -20, and -30 are 10, 20, and 30 respectively. Their sum is **60**.
